# utilities

> Utilities used in measuring the ICL in images of galaxy clusters.

In [ ]:
# | default_exp utilities

In [ ]:
# | export
import numpy as np
from astropy import units as u
from astropy.cosmology import FlatLambdaCDM
from astropy.nddata import CCDData, Cutout2D
from astropy.coordinates import SkyCoord
from astropy.io.fits import Header
from astropy.stats import mad_std, sigma_clip
from astropy.wcs import WCS
from scipy.interpolate import RegularGridInterpolator
from spherical_geometry.polygon import SphericalPolygon


from nicl import ezgal

In [ ]:
# | export


def calc_kcorr(z, filter):
    """Calculate the kcorrection from restframe B-band to an observation in `filter`.

    To convert the conventional B-band surface brightness limit to the observed band we
    use [ezgal](https://github.com/cmancone/easyGalaxy). As ezgal does not work correctly
    when installed via pip, and needs some data files and tweaks, we include a copy with nicl.
    """
    model = ezgal.model("bc03_ssp_z_0.02_chab.model")
    model.set_cosmology(Om=0.3, Ol=0.7, h=0.70, w=-1)
    zf = 3  # Redshift of formation, as in Furnell+21
    rest_b = model.get_absolute_mags(zf=zf, filters="B.dat", zs=z)
    obs_i = model.get_observed_absolute_mags(zf=zf, filters=filter, zs=z)
    kcorr = obs_i - rest_b
    return kcorr * u.mag


def calc_sb_threshold(z, filter, b_band_sb_threshold=25 * u.ABmag):
    k_corr = calc_kcorr(z, filter)
    dimming = 2.5 * np.log10((1 + z) ** 4) * u.mag
    return b_band_sb_threshold + k_corr + dimming


def sb_to_adu(sb, pix_scale, zp=23.9):
    mag = sb - 2.5 * np.log10(pix_scale.to_value(u.arcsec) ** 2)
    counts = 10 ** (-(mag - zp) / 2.5)
    return counts


def adu_to_sb(adu, pix_scale, zp=23.9):
    sb = -2.5 * np.log10(adu / pix_scale**2) + zp
    return sb

In [ ]:
# | export


def get_pixel_scale(img=None, wcs=None):
    """Calculate the average pixel scale of the supplied image or wcs."""
    if wcs is None:
        wcs = img.wcs
    return (
        np.mean([s.to_value("arcsec") for s in wcs.celestial.proj_plane_pixel_scales()])
        * u.arcsec
    )


def get_img_centre_pixel(img):
    return (np.array(img.shape) - 1) / 2


def get_img_centre_world(img, wcs=None):
    """Calculate the world coordinates at the centre of the supplied image."""
    centre_pixel = get_img_centre_pixel(img)
    if wcs is None:
        wcs = img.wcs
    return wcs.pixel_to_world(*centre_pixel)


def distance_from_coord(shape, coord):
    """Calculate the distance from pixel `coord` to each pixel in an image with given `shape`."""
    ij = np.indices(shape)[::-1]
    coord = np.asarray(coord)[:, None, None]
    r = np.sqrt(((ij - coord) ** 2).sum(axis=0))
    return r

In [ ]:
# | export


def physical_to_angular(r_physical, z):
    """Convert a physical distance in the plane of the sky to an angle, for a given redshift `z`."""
    cosmo = FlatLambdaCDM(H0=70, Om0=0.3)
    deg_per_Mpc = 1 / cosmo.kpc_proper_per_arcmin(z).to(u.Mpc / u.deg)
    r_angular = r_physical * deg_per_Mpc
    return r_angular.to(u.deg)

In [ ]:
# | export


def pix2arcmin(
    axis: str,  # the plot axis, 'x' or 'y'
    img: CCDData,  # the image previously displayed with imshow
):
    """Return closures for transforming between pixel coordinates and arcmin offsets.

    These are for use when adding secondary axes to a matplotlib plot created with imshow, e.g.
    `ax.secondary_xaxis('bottom', functions=pix2arcmin('x', img))`.

    The pixel coordinates correspond to the use of `plt.imshow` without specifying an extent.
    Offsets are from the centre of the image.
    """
    idx = 0 if axis == "y" else 1
    pixel_size = get_pixel_scale(img)
    centre = get_img_centre_pixel(img)

    def forward(pix):
        return ((pix - centre[axis]) * pixel_size).to_value(u.arcmin)

    def inverse(arcmin):
        return ((arcmin * u.arcmin / pixel_size) + centre[idx]).to_value()

    return forward, inverse


def pix2Mpc(axis, img, z):
    """Return closures for transforming between pixel coordinates and Mpc offsets.

    These are for use when adding secondary axes to a matplotlib plot created with `plt.imshow`, e.g.
    `ax.secondary_xaxis('bottom', functions=pix2Mpc('x', img, z))`.

    The pixel coordinates correspond to the use of `plt.imshow` without specifying an extent.
    Offsets are from the centre of the image.
    """
    p2a, a2p = pix2arcmin(axis, img)
    arcmin_per_Mpc = physical_to_angular(1 * u.Mpc, z).to_value(u.arcmin)

    def forward(pix):
        return p2a(pix) / arcmin_per_Mpc

    def inverse(Mpc):
        return a2p(Mpc * arcmin_per_Mpc)

    return forward, inverse

In [ ]:
# | export


def get_cutout(image, size, position=None, mask=None, wcs=None):
    if position is None:
        position = get_img_centre_pixel(image).astype(int)
    if wcs is None:
        wcs = image.wcs
    cutout = Cutout2D(image, position, size, wcs=wcs, mode="partial").data
    if mask is not None:
        mask = Cutout2D(mask, position, size, wcs=wcs, mode="partial").data
        return cutout, mask
    else:
        return cutout

In [ ]:
# | export


def maybe_to_value(x, unit):
    """Convert `x` to a value in specified `unit`, assuming in correct unit if `x` is not q Quantity."""
    return u.Quantity(x, unit).to_value(unit)

In [ ]:
# | export


def parse_input_for_skycoord(skycoord):
    """Parse input to return a SkyCoord object.

    Parameters
    ----------
    skycoord : str or SkyCoord
        The input sky coordinate which can be either a string or a SkyCoord object.

    Returns
    -------
    SkyCoord
        A SkyCoord object representing the input sky coordinate.

    """
    if isinstance(skycoord, str):
        return SkyCoord(skycoord, unit=(u.hourangle, u.deg))
    elif isinstance(skycoord, SkyCoord):
        return skycoord
    else:
        raise ValueError("Input must be a string or a SkyCoord object.")


def parse_input_for_angular_size(angular_size, duplicate=False):
    """Parse input to return an angular size Quantity.

    Parameters
    ----------
    angular_size : Quantity, list, tuple, ndarray, or str
        The input angular size which can be a Quantity object, a list, tuple,
        ndarray of values, or a string representing the angular size.
    duplicate : bool or int, optional
        If True or a number equivalent to True, the function will return a list
        containing duplicated angular size values. The total number of elements
        will be `duplicate + 1`. True is equal to 1 in this case. Only applicable
        if the input is a scalar. Useful for turning a single angular size into
        a pair or more. Default is False.

    Returns
    -------
    Quantity or list of Quantity
        The parsed angular size as a Quantity object. If `duplicate` is True,
        returns a list of duplicated Quantity objects.

    """
    # note that u.Quantity is also an instance of ndarray
    if isinstance(angular_size, np.ndarray):
        angular_size = np.atleast_1d(angular_size)
        n_elem = angular_size.size
        match n_elem:
            case 1:
                angular_size = u.Quantity(angular_size[0], unit=u.deg)
            case x if x > 1:
                return [u.Quantity(x, unit=u.deg) for x in angular_size.flat]
            case 0:
                raise ValueError("Input must contain at least one element.")
    elif isinstance(angular_size, (list, tuple)):
        n_elem = len(angular_size)
        match n_elem:
            case 1:
                angular_size = u.Quantity(angular_size[0], unit=u.deg)
            case x if x > 1:
                return [u.Quantity(x, unit=u.deg) for x in angular_size]
            case 0:
                raise ValueError("Input must contain at least one element.")
    elif isinstance(angular_size, (int, float, str)):
        # unit=u.deg is safe because it does not make a difference if the input string has a unit
        angular_size = u.Quantity(angular_size, unit=u.deg)
    else:
        raise ValueError(
            "Input must be a string/int/float, a list/tuple/ndarray, or a Quantity object."
        )
    if duplicate:
        return (int(duplicate) + 1) * [angular_size]
    else:
        return angular_size

In [ ]:
# | export


def create_sky_rectangle(center, width, height, angle=0 * u.deg):
    """Create a rectangle sky region using SphericalPolygon.

    Parameters
    ----------
    center : SkyCoord
        The center of the rectangle
    width : Quantity
        The width of the rectangle
    height : Quantity
        The height of the rectangle
    angle : Quantity, optional
        The rotation angle of the rectangle, default to 0 degrees

    Returns
    -------
    SphericalPolygon
        The rectangular region on the sky as a `SphericalPolygon` object.

    """
    # Convert angle to radians
    angle_rad = angle.to(u.rad).value

    # Calculate half width and height
    half_width = (width / 2.0).to(u.rad).value
    half_height = (height / 2.0).to(u.rad).value

    # Get center coordinates
    ra0, dec0 = center.ra.rad, center.dec.rad

    # Create corner coordinates in the local coordinate system
    corners_local = np.array(
        [
            [half_width, half_height],  # Northeast
            [-half_width, half_height],  # Northwest
            [-half_width, -half_height],  # Southwest
            [half_width, -half_height],  # Southeast
        ]
    )

    # Create rotation matrix
    cos_angle = np.cos(angle_rad)
    sin_angle = np.sin(angle_rad)
    rotation_matrix = np.array([[cos_angle, -sin_angle], [sin_angle, cos_angle]])

    # Apply rotation to all corners at once
    corners_rotated = np.dot(corners_local, rotation_matrix.T)
    # Calculate angular separations and PAs for all corners
    separations = (
        np.sqrt(corners_rotated[:, 0] ** 2 + corners_rotated[:, 1] ** 2) * u.rad
    )
    position_angles = np.arctan2(corners_rotated[:, 0], corners_rotated[:, 1]) * u.rad
    center_sky = SkyCoord(ra=ra0 * u.rad, dec=dec0 * u.rad)
    corners = center_sky.directional_offset_by(position_angles, separations)

    # Extract RA and Dec values
    ras = corners.ra.rad
    decs = corners.dec.rad
    # Explicitly wrap RA to handle boundary crossings
    ras = np.mod(ras, 2 * np.pi)

    # Create the SphericalPolygon (the construction method automatically appends first point at the end to close the polygon)
    return SphericalPolygon.from_radec(ras, decs, center=(ra0, dec0), degrees=False)


def does_image_overlap_with_skyregion(hdr, sky_reg):
    """Check if an image overlaps with a sky region.

    Parameters
    ----------
    hdr : dict-like
        The FITS header of the image to check for overlap with the sky region.
        Must be compatible with WCS creation and refer to a 2D image.
    sky_reg : SphericalPolygon
        The sky region to check for overlap with the image.
    threshold : float, optional
        Minimum fraction of overlap required to return True. Default is 0.0,
        which means any overlap will return True.

    Returns
    -------
    bool
        True if the image overlaps with the sky region by more than the threshold,
        False otherwise.

    """
    # Check if header is a dict-like object
    if not isinstance(hdr, (dict, Header)) or not hasattr(hdr, "keys"):
        raise ValueError("Input header must be a dict-like object")

    # Check if the image is 2D
    if "NAXIS" not in hdr:
        raise ValueError("Header doesn't contain NAXIS keyword")
    if hdr["NAXIS"] != 2:
        raise ValueError("This function only works with 2D images (NAXIS=2)")
    if "NAXIS1" not in hdr or "NAXIS2" not in hdr:
        raise ValueError(
            "Header must contain NAXIS1 and NAXIS2 keywords for a 2D image"
        )
    # Check if the input sky region has the correct type
    if not isinstance(sky_reg, SphericalPolygon):
        raise ValueError("sky_reg must be a SphericalPolygon object.")

    sky_reg_from_img = SphericalPolygon.from_wcs(hdr, steps=2)
    return sky_reg.intersects_poly(sky_reg_from_img)


def compute_pixel_scales(hdr, accuracy=1e-4, initial_grid_step=None):
    """Compute effective pixel-scale map from FITS WCS.

    Compute the geometric mean of the local x- and y-direction pixel scales
    (in arcsec/pixel) at each image pixel by interpolating scales on a sparse grid
    until the specified fractional accuracy is reached.

    Parameters
    ----------
    hdr : astropy.io.fits.Header or dict-like
        FITS header with valid 2D WCS (NAXIS1, NAXIS2).
    accuracy : float, optional
        Target fractional accuracy for pixel-scale interpolation (default=1e-4).
    initial_grid_step : int, optional
        Initial grid spacing for scale computation; default is half the smaller image dimension.

    Returns
    -------
    numpy.ndarray
        2D array of effective pixel scales (arcsec per pixel) for the full image.

    """
    wcs = WCS(hdr)
    ny, nx = hdr["NAXIS2"], hdr["NAXIS1"]

    if initial_grid_step is None:
        initial_grid_step = min(ny, nx) // 2
    if initial_grid_step <= 1:
        raise ValueError("Initial grid step must be 2 or more for the first iteration.")

    # Function to compute scales on a grid with given step size
    def compute_grid_scales(grid_step, delta=0.5):
        # Create a grid for computing scales
        y_sparse = np.arange(0, ny, grid_step)
        x_sparse = np.arange(0, nx, grid_step)

        # Ensure the grid includes the last pixel in each dimension
        if x_sparse[-1] != nx - 1:
            x_sparse = np.append(x_sparse, nx - 1)
        if y_sparse[-1] != ny - 1:
            y_sparse = np.append(y_sparse, ny - 1)

        # Create grid meshes
        yy_sparse, xx_sparse = np.meshgrid(y_sparse, x_sparse, indexing="ij")

        # Prepare coordinate arrays for vectorized computation
        coords = np.stack([xx_sparse.ravel(), yy_sparse.ravel()], axis=1).astype(float)

        # Create offset coordinates in batch
        coords_plus_x = coords.copy()
        coords_plus_x[:, 0] += delta

        coords_minus_x = coords.copy()
        coords_minus_x[:, 0] -= delta

        coords_plus_y = coords.copy()
        coords_plus_y[:, 1] += delta

        coords_minus_y = coords.copy()
        coords_minus_y[:, 1] -= delta

        # Batch compute world coordinates
        world_plus_x = wcs.all_pix2world(coords_plus_x, 0)
        world_minus_x = wcs.all_pix2world(coords_minus_x, 0)
        world_plus_y = wcs.all_pix2world(coords_plus_y, 0)
        world_minus_y = wcs.all_pix2world(coords_minus_y, 0)

        # Compute scales in arcsec/pixel
        # Use astropy SkyCoord to handle spherical geometry correctly
        ra1, dec1 = world_minus_x[:, 0], world_minus_x[:, 1]
        ra2, dec2 = world_plus_x[:, 0], world_plus_x[:, 1]
        coords1 = SkyCoord(ra1, dec1, unit="deg")
        coords2 = SkyCoord(ra2, dec2, unit="deg")
        scale_x = coords1.separation(coords2).arcsec / (2 * delta)

        ra1, dec1 = world_minus_y[:, 0], world_minus_y[:, 1]
        ra2, dec2 = world_plus_y[:, 0], world_plus_y[:, 1]
        coords1 = SkyCoord(ra1, dec1, unit="deg")
        coords2 = SkyCoord(ra2, dec2, unit="deg")
        scale_y = coords1.separation(coords2).arcsec / (2 * delta)

        # Effective scale (geometric mean)
        scales = np.sqrt(scale_x * scale_y).reshape(len(y_sparse), len(x_sparse))

        return scales, y_sparse, x_sparse

    # Iteratively refine grid step to achieve required accuracy
    scales_old, y_sparse_old, x_sparse_old = compute_grid_scales(initial_grid_step)
    iter_count = 0

    while True:  # Will break inside the loop when accuracy is reached
        if iter_count == 0:
            grid_step = initial_grid_step // 2
        scales_new, y_sparse_new, x_sparse_new = compute_grid_scales(grid_step)

        # Interpolate old scales to match new grid for comparison
        old_interp = RegularGridInterpolator(
            (y_sparse_old, x_sparse_old),
            scales_old,
            method="linear",
            bounds_error=False,
            fill_value=None,
        )

        # Create points for interpolation using the actual sparse coordinates
        yy_sparse, xx_sparse = np.meshgrid(y_sparse_new, x_sparse_new, indexing="ij")
        points = np.column_stack((yy_sparse.ravel(), xx_sparse.ravel()))

        # Interpolate old scales to new grid
        scales_old_interp = old_interp(points).reshape(scales_new.shape)

        # Calculate relative difference
        rel_diff = np.abs(scales_new - scales_old_interp) / scales_new

        # Check if we've reached desired accuracy or minimum grid step
        if np.all(rel_diff <= accuracy):
            # print(f"Converged after {iter_count+1} iterations.")
            break
        elif grid_step <= 1:
            print("Reached minimum grid step, breaking the loop.")
            break

        # For TPV projections with distortions, a more adaptive approach is better
        # This combines binary search with a safety factor to avoid oscillations
        mean_rel_diff = np.abs(np.mean(scales_new - scales_old_interp)) / np.mean(
            scales_new
        )
        if mean_rel_diff > 4 * accuracy:  # Far from target - be aggressive
            grid_step = max(1, grid_step // 2)
        elif mean_rel_diff > 2 * accuracy:  # Getting closer
            grid_step = max(1, int(grid_step * 0.6))
        else:  # Close to target - fine adjustments
            grid_step = max(1, int(grid_step * 0.8))

        # Maximum iterations safety check
        if iter_count >= 100:  # Add iteration counter for safety
            print("Maximum iterations (100) reached, breaking the loop.")
            break

        scales_old = scales_new
        y_sparse_old = y_sparse_new
        x_sparse_old = x_sparse_new
        iter_count += 1  # Increment iteration counter

    # Create interpolator for final grid
    interp = RegularGridInterpolator(
        (y_sparse_new, x_sparse_new),
        scales_new,
        method="linear",
        bounds_error=False,
        fill_value=None,
    )

    # Generate full coordinate grid
    y_full, x_full = np.mgrid[0:ny, 0:nx]
    points = np.column_stack((y_full.ravel(), x_full.ravel()))

    # Interpolate to get full grid of pixel scales
    pixel_scale_eff = interp(points).reshape(ny, nx)

    return pixel_scale_eff

In [ ]:
# | export


def sigma_clip_stats(
    x, sigma=3, sigma_lower=None, sigma_upper=None, axis=None, maxiters=10, ngood_min=1
):
    """Calculate statistics on sigma-clipped data.

    Replacement for astropy.stats.sigma_clipped_stats. This function is a wrapper
    around astropy.stats.sigma_clip that performs sigma clipping on the input data and
    returns common statistics calculated on the clipped data.

    Parameters
    ----------
    x : array-like
        The data to be sigma clipped.
    sigma : float, optional
        The number of standard deviations to use for both the lower and upper
        clipping limit. Default is 3.
    sigma_lower : float or None, optional
        The number of standard deviations to use as the lower limit. If None,
        the value of `sigma` is used. Default is None.
    sigma_upper : float or None, optional
        The number of standard deviations to use as the upper limit. If None,
        the value of `sigma` is used. Default is None.
    axis : int or None, optional
        The axis along which to sigma clip. If None, the flattened array is used.
        Default is None.
    maxiters : int, optional
        The maximum number of sigma clipping iterations to perform. Default is 10.
    ngood_min : int, optional
        The minimum number of good (non-clipped) data points required to return
        valid statistics. If fewer points remain, NaN values are returned. Default is 1.

    Returns
    -------
    mean : float or ndarray
        The mean of the sigma-clipped data.
    median : float or ndarray
        The median of the sigma-clipped data.
    std : float or ndarray
        The standard deviation of the sigma-clipped data.
    mean_err : float or ndarray
        The standard error of the mean (std/sqrt(n)) of the sigma-clipped data.

    """
    clipped = sigma_clip(
        x,
        sigma=sigma,
        sigma_lower=sigma_lower,
        sigma_upper=sigma_upper,
        axis=axis,
        maxiters=maxiters,
        masked=False,
    )
    ngood = np.isfinite(clipped).sum(axis=axis)
    mean = np.nanmean(clipped, axis=axis)
    median = np.nanmedian(clipped, axis=axis)
    std = np.nanstd(clipped, axis=axis)
    mean_err = std / np.sqrt(ngood)
    if ngood.ndim < 1:
        if ngood < ngood_min:
            return np.nan, np.nan, np.nan, np.nan
        else:
            return mean, median, std, mean_err
    idx = ngood < ngood_min
    mean[idx] = np.nan
    median[idx] = np.nan
    std[idx] = np.nan
    mean_err[idx] = np.nan
    return mean, median, std, mean_err


def weighted_median(values, weights):
    sorter = np.argsort(values)
    values_sorted = values[sorter]
    weights_sorted = weights[sorter]
    cum_weights = np.cumsum(weights_sorted)
    midpoint = 0.5 * np.sum(weights_sorted)
    median_idx = np.searchsorted(cum_weights, midpoint)
    return values_sorted[median_idx]


def bootstrap_median_error(data, errors, nboot=1000, axis=None):
    """Compute the bootstrap estimate of the median's standard error.

    Generates `nboot` synthetic datasets by sampling each element of `data` from a normal
    distribution defined by the corresponding `errors`, computes the median over each
    bootstrap realization along `axis`, and returns the standard deviation of those medians
    as the uncertainty.

    Parameters
    ----------
    data : array-like
        Input values for median computation. Scalar or array.
    errors : array-like
        1-sigma uncertainties for each element in `data`; same shape as `data`.
    nboot : int, optional
        Number of bootstrap realizations (default=1000).
    axis : int or None, optional
        Axis along which to compute medians. If None, data is flattened (default).

    Returns
    -------
    float or ndarray
        Bootstrap-derived standard error of the median. Scalar if `axis` is None,
        or array with `axis` dimension removed.

    Notes
    -----
    - If the specified `axis` has length 1, the original `errors` value is returned.
    - If `axis` is None and `data.size <= 1`, returns the single `errors` value.

    """
    data = np.asarray(data)
    errors = np.asarray(errors)

    # If axis is not None and the shape along that axis is 1, return the errors directly
    if axis is not None and data.shape[axis] == 1:
        # Need to squeeze out the singular dimension
        return np.squeeze(errors, axis=axis)
    # If axis is None, check if they have more than one element
    elif axis is None and data.size <= 1:
        return np.squeeze(errors)

    # Create all synthetic bootstrap samples at once.
    # The simulated array will have shape (nboot, ...) where ... is data.shape.
    simulated = np.random.normal(loc=data, scale=errors, size=(nboot,) + data.shape)

    if axis is None:
        # Flatten each bootstrap sample and compute the median along the flattened axis.
        medians = np.nanmedian(simulated.reshape(nboot, -1), axis=1)
    else:
        # Adjust the axis by 1 because the bootstrap dimension is at index 0.
        medians = np.nanmedian(simulated, axis=axis + 1)

    # Compute the standard deviation of the medians along the bootstrap dimension.
    return np.nanstd(medians, axis=0)


def fast_median_error(data, axis=None):
    """Compute analytical median error using a normal approximation.

    The median error is estimated as 1.253 * MAD / sqrt(n), where MAD is
    the median absolute deviation and n is the number of finite samples.

    Parameters
    ----------
    data : array-like
        Input values for median error estimation.
    axis : int or None, optional
        Axis along which to compute the median error. If None, data is flattened.

    Returns
    -------
    float or ndarray
        Estimated standard error of the median. Scalar if `axis` is None or the
        selected axis has length 1, otherwise an array with that axis removed.

    Raises
    ------
    ValueError
        If the specified axis has length 1 or data size <= 1.

    """
    data = np.asarray(data)

    # If axis is not None and the shape along that axis is 1, raise an error
    if axis is not None and data.shape[axis] == 1:
        raise ValueError(
            "Cannot compute median error along axis with only one element."
        )
    # If axis is None, check if they have more than one element
    elif axis is None and data.size <= 1:
        raise ValueError(
            "Cannot compute median error for an array with only one element."
        )

    # Compute the standard error of the median using a normal approximation 1.253*sigma/sqrt(n)
    return (
        1.253
        * mad_std(data, axis=axis, ignore_nan=True)
        / np.sqrt(np.isfinite(data).sum(axis=axis))
    )

In [ ]:
# | hide
import nbdev

nbdev.nbdev_export()